# 02 Vegetation Condition

**Project:** Pine Ridge Bison Habitat Suitability Analysis  
**BHSI Component:** Vegetation (30% weight)  
**Data Sources:** MODIS MOD13Q1 (NDVI), NLCD 2021 (land cover)

## What This Notebook Does
Vegetation is the dominant factor in bison habitat suitability as it
determines what bison eat and how much land can support.
This notebook builds the vegetation suitability layer from two sources:

**NLCD 2021 land cover**: assigns a base suitability score by land cover
class. Native grassland (class 71) receives the highest score; cropland
the lowest. This reflects the current state of the land.

**MODIS NDVI trend**: adjusts the NLCD base score based on the long-term
vegetation trend (2000–2023). Land with improving NDVI gets a bonus;
declining NDVI gets a penalty. This captures trajectory, not just current
state, which is important for identifying restoration opportunity.

## Bison Forage Requirements
Bison are grazers, they consume primarily grasses and forbs.
On the Northern Great Plains, the mixed-grass prairie that dominates
Pine Ridge provides the nutritional diversity they need year-round.
Key species include blue grama, western wheatgrass, buffalograss,
and needle-and-thread grass. Land in agricultural production (crops)
cannot support bison without restoration to native or managed grassland.

In [ ]:
# Imports
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio
from rasterio.warp import reproject, Resampling
from scipy import stats

# Uncomment these lines if needed to install dependencies (run once)
# import subprocess
# subprocess.run(["pip", "install", "planetary-computer", "pystac-client", "odc-stac", "--quiet"])

import planetary_computer
import pystac_client
import odc.stac
import numpy as np
import rasterio
from pathlib import Path
from src.constants import PINE_RIDGE_BBOX, CACHE_DIR
from affine import Affine

from src.constants import (
    CRS_GEOGRAPHIC, CRS_PROJECTED, TARGET_RES_M,
    PINE_RIDGE_BBOX, PINE_RIDGE_LAT, PINE_RIDGE_LON,
    MODIS_START_YEAR, MODIS_END_YEAR, GROWING_MONTHS,
    NLCD_BISON_SUITABILITY,
    CACHE_DIR, OUTPUTS_DIR, FIGURES_DIR,
)
from src.loaders import fetch_ndvi_point, load_nlcd_bbox
from src.raster_utils import (
    normalize_0_1,
    align_raster_to_template,
)
from src.sovereignty import print_data_acknowledgment, generate_citations

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
%matplotlib inline

TEMPLATE_PATH = CACHE_DIR / "template_30m_albers.tif"
assert TEMPLATE_PATH.exists(), "Run notebook 01 first to create the template raster."

boundary_path = OUTPUTS_DIR / "pine_ridge_boundary.geojson"
pine_ridge    = gpd.read_file(boundary_path)
print("Setup complete.")

In [ ]:
# Print the data sovereignty statement at the top of every notebook
print_data_acknowledgment(source_keys=["modis_ndvi", "nlcd"])

In [ ]:
# Set parameters for LCMAP

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

lcmap_items = catalog.search(
    collections = ["usgs-lcmap-conus-v13"],
    bbox        = list(PINE_RIDGE_BBOX),
    datetime    = "2021-01-01/2021-12-31",
).item_collection()

print(f"LCMAP items: {len(lcmap_items)}")

In [ ]:
# Load LCMAP primary land cover for Pine Ridge
lcmap_ds = odc.stac.load(
    lcmap_items,
    bands      = ["lcpri"],
    bbox       = list(PINE_RIDGE_BBOX),
    crs        = "EPSG:4326",
    resolution = 0.0003,
)

nlcd_data      = lcmap_ds["lcpri"].values[0].astype(np.uint8)
nlcd_transform = lcmap_ds["lcpri"].odc.geobox.transform
nlcd_crs       = "EPSG:4326"

print(f"Shape         : {nlcd_data.shape}")
print(f"Unique classes: {sorted(np.unique(nlcd_data).tolist())}")

In [ ]:
# LCMAP v1.3 primary land cover class for bison suitability score
# Reference: https://www.usgs.gov/media/files/lcmap-collection-13-product-guide
LCMAP_BISON_SUITABILITY = {
    1:  ("Developed",               0.00),
    2:  ("Cropland",                0.10),
    3:  ("Grass/Shrub",             0.85),   # mixed-grass prairie — bison ideal
    4:  ("Tree Cover",              0.15),
    5:  ("Water",                   0.00),
    6:  ("Wetland",                 0.45),
    7:  ("Ice/Snow",                0.00),
    8:  ("Barren",                  0.15),
}

# Override the BISON_SUITABILITY lookup in the notebook
# Apply the same classification logic
suit_array = np.zeros_like(nlcd_data, dtype=np.float32)
suit_array[:] = np.nan

class_coverage = {}
for class_val, (class_name, score) in LCMAP_BISON_SUITABILITY.items():
    mask = nlcd_data == class_val
    suit_array[mask] = score
    n_px = int(mask.sum())
    if n_px > 0:
        class_coverage[class_name] = {"pixels": n_px, "score": score, "class": class_val}

print("LCMAP LAND COVER ON PINE RIDGE")
total_px = (~np.isnan(suit_array)).sum()
for name, info in sorted(class_coverage.items(), key=lambda x: -x[1]["pixels"]):
    pct = info["pixels"] / total_px * 100 if total_px > 0 else 0
    bar = "█" * int(pct / 2)
    print(f"  [{info['class']}] {name:<25} score={info['score']:.2f}  {pct:>5.1f}%  {bar}")

In [ ]:
# Save LCMAP-derived suitability raster and continue with the rest of notebook 02
from src.raster_utils import align_raster_to_template

nlcd_suit_raw = CACHE_DIR / "nlcd_suit_raw.tif"
profile = {
    "driver": "GTiff", "dtype": "float32", "count": 1,
    "crs": nlcd_crs, "transform": nlcd_transform,
    "width": nlcd_data.shape[1], "height": nlcd_data.shape[0],
    "nodata": np.nan, "compress": "lzw",
}
with rasterio.open(nlcd_suit_raw, "w", **profile) as dst:
    dst.write(suit_array, 1)

nlcd_suit_aligned = CACHE_DIR / "nlcd_suit_aligned.tif"
align_raster_to_template(
    src_path=nlcd_suit_raw,
    template_path=TEMPLATE_PATH,
    output_path=nlcd_suit_aligned,
    resampling_method="nearest",
)
print("LCMAP suitability raster aligned to template continuing with notebook 02.")
print("Note: LCMAP class 3 (Grass/Shrub) covers both grassland and shrubland.")
print("On Pine Ridge this is predominantly mixed-grass prairie — correct for bison.")

## LCMAP Land Cover Baseline Suitability

In [ ]:
# Reuse the LCMAP classification above; do not apply NLCD class codes to LCMAP values.
suit_array = np.zeros_like(nlcd_data, dtype=np.float32)
suit_array[:] = np.nan   # default = no score (water, developed)

class_coverage = {}
for class_val, (class_name, score) in LCMAP_BISON_SUITABILITY.items():
    mask = nlcd_data == class_val
    suit_array[mask] = score
    n_px = int(mask.sum())
    if n_px > 0:
        class_coverage[class_name] = {
            "pixels":  n_px,
            "score":   score,
            "class":   class_val,
        }

# Coverage summary
print("LAND COVER ON PINE RIDGE")
total_px = (~np.isnan(suit_array)).sum()
for name, info in sorted(class_coverage.items(),
                          key=lambda x: -x[1]["pixels"]):
    pct = info["pixels"] / total_px * 100 if total_px > 0 else 0
    bar = "█" * int(pct / 2)
    print(f"  [{info['class']:>2}] {name:<35} "
          f"score={info['score']:.2f}  {pct:>5.1f}%  {bar}")

In [ ]:
# Save LCMAP-derived suitability raster and continue with the rest of notebook 02
from src.raster_utils import align_raster_to_template

nlcd_suit_raw = CACHE_DIR / "nlcd_suit_raw.tif"
profile = {
    "driver": "GTiff", "dtype": "float32", "count": 1,
    "crs": nlcd_crs, "transform": nlcd_transform,
    "width": nlcd_data.shape[1], "height": nlcd_data.shape[0],
    "nodata": np.nan, "compress": "lzw",
}
with rasterio.open(nlcd_suit_raw, "w", **profile) as dst:
    dst.write(suit_array, 1)

nlcd_suit_aligned = CACHE_DIR / "nlcd_suit_aligned.tif"
align_raster_to_template(
    src_path=nlcd_suit_raw,
    template_path=TEMPLATE_PATH,
    output_path=nlcd_suit_aligned,
    resampling_method="nearest",
)
print("LCMAP suitability raster aligned to template, continuing with notebook 02.")
print("Note: LCMAP class 3 (Grass/Shrub) covers both grassland and shrubland.")
print("On Pine Ridge this is predominantly mixed-grass prairie which is correct for bison.")

## MODIS NDVI Long-Term Trend Adjustment

In [ ]:
# Fetch NDVI for the Pine Ridge centroid
# 24 years × 3 API chunks/year = 72 requests for first run (might take some time)

ndvi_df = fetch_ndvi_point(
    lat=PINE_RIDGE_LAT, lon=PINE_RIDGE_LON,
    start_year=MODIS_START_YEAR, end_year=MODIS_END_YEAR,
    site_name="pine_ridge_centroid",
)

print(f"NDVI records: {len(ndvi_df):,}")
print(f"Date range  : {ndvi_df['date'].min().date()} to "
      f"{ndvi_df['date'].max().date()}")
print(f"NDVI range  : {ndvi_df['ndvi'].min():.3f} – {ndvi_df['ndvi'].max():.3f}")

In [ ]:
# Compute annual growing season mean NDVI and Theil-Sen trend
ndvi_df["year"]  = ndvi_df["date"].dt.year
ndvi_df["month"] = ndvi_df["date"].dt.month

ndvi_gs = ndvi_df[ndvi_df["month"].isin(GROWING_MONTHS)]
annual_ndvi = (
    ndvi_gs.groupby("year")["ndvi"]
    .mean()
    .reset_index()
)

# Theil-Sen slope
yrs  = annual_ndvi["year"].values.astype(float)
vals = annual_ndvi["ndvi"].values
slope, intercept, _, _ = stats.theilslopes(vals, yrs)
_, _, _, p, _          = stats.linregress(yrs, vals)

print("GROWING SEASON NDVI TREND for the Pine Ridge Centroid")
print(f"  Long-term mean NDVI  : {vals.mean():.4f}")
print(f"  Theil-Sen slope      : {slope*10:+.4f} NDVI units/decade")
print(f"  Significance         : p = {p:.3f} "
      f"({'significant' if p < 0.05 else 'not significant'})")
print(f"  Direction            : "
      f"{'Improving ↑' if slope > 0 else 'Declining ↓'}")

In [ ]:
# The NDVI trend adjusts the NLCD base score
# This is a point-based proxy. In a full spatial analysis, per-pixel
# NDVI trends from MODIS gridded data would be used

# Trend adjustment factor: slope normalized to [-0.15, +0.15] range
# i.e., a strong improving trend adds up to 15% to the NLCD score
TREND_MAX_ADJUST = 0.15

# Normalize slope to ±TREND_MAX_ADJUST
# Reference: ±0.05 NDVI/decade = typical range for grassland trend
trend_adjust = np.clip(
    slope * 10 / 0.05 * TREND_MAX_ADJUST,
    -TREND_MAX_ADJUST, TREND_MAX_ADJUST
)

print(f"NDVI trend adjustment factor: {trend_adjust:+.3f}")
print(f"Applied uniformly to all NLCD suitability scores.")
print()
print("Note: A full raster-scale analysis would compute per-pixel")
print("NDVI trends from the MODIS 250m gridded product. The centroid")
print("proxy used here is appropriate for the reservation-scale BHSI.")

In [ ]:
# Apply trend adjustment to the aligned NLCD suitability raster
with rasterio.open(nlcd_suit_aligned) as src:
    veg_data    = src.read(1).astype(np.float32)
    veg_profile = src.profile.copy()

# Apply trend adjustment and clip to [0, 1]
veg_adjusted = np.where(
    ~np.isnan(veg_data),
    np.clip(veg_data + trend_adjust, 0, 1),
    np.nan,
).astype(np.float32)

# Write final vegetation suitability layer
veg_suit_path = OUTPUTS_DIR / "bhsi_vegetation.tif"
with rasterio.open(veg_suit_path, "w", **veg_profile) as dst:
    dst.write(veg_adjusted, 1)

valid_veg = veg_adjusted[~np.isnan(veg_adjusted)]
print(f"Vegetation suitability layer:")
print(f"  File  : outputs/bhsi_vegetation.tif")
print(f"  Min   : {valid_veg.min():.3f}")
print(f"  Max   : {valid_veg.max():.3f}")
print(f"  Mean  : {valid_veg.mean():.3f}")
print(f"  Pixels: {len(valid_veg):,}")

---
## Visualizations

In [ ]:
def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# NDVI time series
ax = axes[0]
ax.scatter(annual_ndvi["year"], annual_ndvi["ndvi"],
           color="#27AE60", s=40, alpha=0.8)
trend_line = slope * yrs + intercept
ax.plot(annual_ndvi["year"], trend_line, color="black",
        linewidth=2, linestyle="--",
        label=f"Trend: {slope*10:+.4f} NDVI/decade (p={p:.3f})")
ax.set_xlabel("Year", fontsize=10)
ax.set_ylabel("Growing season mean NDVI", fontsize=10)
ax.set_title(
    "NDVI Trend for Pine Ridge (2000–2023)\n"
    "Trend adjustment applied to vegetation suitability layer",
    fontsize=10, fontweight="bold",
)
ax.legend(fontsize=9)
despine(ax)

# Vegetation suitability raster
ax = axes[1]
im = ax.imshow(
    veg_adjusted, cmap="YlGn",
    vmin=0, vmax=1, origin="upper",
)
plt.colorbar(im, ax=ax, label="Vegetation suitability (0–1)", shrink=0.8)
ax.set_title(
    "Vegetation Suitability Layer\n"
    "NLCD base score + NDVI trend adjustment",
    fontsize=10, fontweight="bold",
)
ax.set_xlabel("Column (west to east)")
ax.set_ylabel("Row (north to south)")

plt.suptitle(
    "BHSI Component 1: Vegetation (30% weight)\n"
    "Pine Ridge Reservation, Oglala Lakota Nation",
    fontsize=12, fontweight="bold",
)
plt.tight_layout()
fig.savefig(FIGURES_DIR/"02_vegetation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print(generate_citations(["modis_ndvi", "nlcd"]))